# ⚙️ 05c — Feature Engineering v3
## Cambios respecto a 05b

1. **Lee `dataset_labeled_v2.parquet`** — con yaw_cable lead=72h
2. **`pitch_bat`**: elimina temperatura ambiente como feature directa — el modelo estaba aprendiendo estacionalidad (invierno=fallo) en lugar de degradación de batería. Se sustituye por `t_motor_vs_ambient` (delta motor-ambiente) que captura si el motor NO está calentando respecto al frío exterior.
3. **Baseline**: mismo cálculo que v2, primeros 180 días


In [21]:
import os, time
import pandas as pd
import numpy as np

base_dir   = os.path.dirname(os.getcwd())
silver_dir = os.path.join(base_dir, 'data', 'silver')

# Lee el dataset con lead times corregidos
df = pd.read_parquet(os.path.join(silver_dir, 'dataset_labeled_v2.parquet'))
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Dataset v2: {len(df):,} filas' {df.columns})
for fam in ['yaw_cable', 'brake_hydro', 'generator', 'pitch_bat']:
    n = df[f'is_pre_{fam}'].sum()
    print(f'  {fam:<15}: {n:>7,} positivos ({100*n/len(df):.1f}%)')


SyntaxError: invalid syntax. Perhaps you forgot a comma? (3280373166.py, line 12)

In [13]:
# ==============================================================================
# FEATURES DE DOMINIO
# ==============================================================================
original_columns = set(df.columns)

# YAW
df['yaw_error']       = (df['nacelle_position'] - df['wind_direction']).abs() % 360
df['yaw_error']       = df['yaw_error'].apply(lambda x: x if x <= 180 else 360 - x)
df['yaw_error_wind']  = df['yaw_error'] * df['wind_speed_ms']
df['cable_rate']      = df['cable_windings_from_calibration_point'].diff(1).fillna(0)
df['nacelle_std_ratio']= df['nacelle_position_standard_deviation'] / (df['wind_speed_ms'] + 1e-6)

# TÉRMICAS GENERADOR
df['t_bearing_delta']      = df['generator_bearing_front_temperature_c'] - df['nacelle_ambient_temperature_c']
df['t_rear_bearing_delta'] = df['generator_bearing_rear_temperature_c']  - df['nacelle_ambient_temperature_c']
df['t_stator_delta']       = df['stator_temperature_1_c']                - df['nacelle_ambient_temperature_c']
df['t_gear_oil_delta']     = df['gear_oil_temperature_c']                - df['nacelle_ambient_temperature_c']
df['t_bearing_diff']       = df['generator_bearing_front_temperature_c'] - df['generator_bearing_rear_temperature_c']
df['t_stator_bearing_diff']= df['stator_temperature_1_c']                - df['generator_bearing_front_temperature_c']
df['t_bearing_delta_roc']  = df['t_bearing_delta'].diff(6)
df['t_stator_roc']         = df['stator_temperature_1_c'].diff(6)

# ELÉCTRICO
df['apparent_power_kva']   = (df['power_kw']**2 + df['reactive_power_kvar']**2) ** 0.5
df['reactive_power_ratio'] = df['reactive_power_kvar'] / (df['apparent_power_kva'] + 1e-6)

# HIDRÁULICO
df['pressure_vs_temp']    = df['gear_oil_inlet_pressure_bar'] / (df['gear_oil_inlet_temperature_c'] + 273.15)
df['metal_particle_rate'] = df['metal_particle_count'].diff(1).fillna(0).clip(lower=0)

# PITCH — NUEVO: deltas motor-ambiente en lugar de temperatura absoluta
# El modelo aprenderá "el motor no calienta respecto al frío" no "es invierno"
df['t_motor1_vs_ambient'] = df['temperature_motor_axis_1_c'] - df['nacelle_ambient_temperature_c']
df['t_motor2_vs_ambient'] = df['temperature_motor_axis_2_c'] - df['nacelle_ambient_temperature_c']
df['t_motor3_vs_ambient'] = df['temperature_motor_axis_3_c'] - df['nacelle_ambient_temperature_c']
df['pitch_asymmetry']     = (df[['blade_angle_pitch_position_a','blade_angle_pitch_position_b','blade_angle_pitch_position_c']].max(axis=1) -
                              df[['blade_angle_pitch_position_a','blade_angle_pitch_position_b','blade_angle_pitch_position_c']].min(axis=1))
df['blade_angle_mean']        = df[['blade_angle_pitch_position_a','blade_angle_pitch_position_b','blade_angle_pitch_position_c']].mean(axis=1)
df['motor_current_imbalance'] = df[['motor_current_axis_1_a','motor_current_axis_2_a','motor_current_axis_3_a']].std(axis=1)

print(f'Features de dominio: {len([c for c in df.columns if c not in original_columns])}')


Features de dominio: 22


In [14]:
# ==============================================================================
# BASELINE (primeros 180 días)
# ==============================================================================
baseline_cutoff = df['timestamp'].min() + pd.Timedelta(days=180)
df_bl = df[df['timestamp'] < baseline_cutoff]

sensor_cols = [c for c in df.columns
               if c not in ['timestamp']
               and not c.startswith('is_pre_')
               and not c.startswith('hours_to_')
               and df[c].dtype in [float, 'float64', 'float32']]

baseline_mean = df_bl[sensor_cols].mean()
baseline_p90  = df_bl[sensor_cols].quantile(0.90)
print(f'Baseline calculado sobre {len(df_bl):,} filas ({baseline_cutoff.date()})')


Baseline calculado sobre 25,926 filas (2018-06-29)


In [19]:
# ==============================================================================
# FUNCIÓN ROLLING v2 (mean, std, p95, exceedance, baseline_ratio)
# ==============================================================================
WINDOWS = {'1h': 6, '6h': 36, '24h': 144, '7d': 1008}

def make_rolling_features_v2(df, sensors, windows, baseline_mean, baseline_p90):
    feats = {}
    for col in sensors:
        if col not in df.columns:
            print(f'  WARN: {col} no encontrada')
            continue
        s = df[col].ffill().fillna(0)
        thresh = baseline_p90.get(col, s.quantile(0.90))
        for wname, w in windows.items():
            mp   = max(1, w // 3)
            roll = s.rolling(w, min_periods=mp)
            feats[f'{col}__mean_{wname}']   = roll.mean()
            feats[f'{col}__std_{wname}']    = roll.std().fillna(0)
            feats[f'{col}__p95_{wname}']    = roll.quantile(0.95)
            feats[f'{col}__exceed_{wname}'] = s.rolling(w, min_periods=mp).apply(
                lambda x: (x > thresh).mean(), raw=True)
        bm = baseline_mean.get(col, 1.0)
        if abs(bm) > 1e-6:
            feats[f'{col}__baseline_ratio'] = s.rolling(WINDOWS['7d'], min_periods=max(1, WINDOWS['7d']//3)).mean() / abs(bm)
    return pd.DataFrame(feats, index=df.index)

print('Función definida')


Función definida


In [20]:
# ==============================================================================
# SENSORES POR FAMILIA
# pitch_bat: sin nacelle_ambient_temperature_c directo → sustituido por deltas
# ==============================================================================
FAMILY_SENSORS = {
    'yaw_cable': [
        'nacelle_position', 'nacelle_position_standard_deviation',
        'wind_direction', 'wind_direction_standard_deviation',
        'vane_position_12', 'cable_windings_from_calibration_point',
        'wind_speed_ms', 'power_kw',
        'yaw_error', 'yaw_error_wind', 'cable_rate', 'nacelle_std_ratio',
    ],
    'brake_hydro': [
        'gear_oil_inlet_pressure_bar', 'gear_oil_pump_pressure_bar',
        'gear_oil_inlet_temperature_c', 'gear_oil_temperature_c',
        'generator_rpm_rpm', 'generator_rpm_standard_deviation_rpm',
        'rotor_speed_rpm', 'power_kw',
        'front_bearing_temperature_c', 'rear_bearing_temperature_c',
        'metal_particle_count',
        't_gear_oil_delta', 'pressure_vs_temp', 'metal_particle_rate',
    ],
    'generator': [
        'generator_bearing_front_temperature_c', 'generator_bearing_rear_temperature_c',
        'generator_bearing_front_temperature_max_c', 'generator_bearing_rear_temperature_max_c',
        'nacelle_temperature_c', 'nacelle_ambient_temperature_c',
        'ambient_temperature_converter_c', 'power_kw', 'reactive_power_kvar',
        'power_factor_cosphi', 'stator_temperature_1_c', 'wind_speed_ms',
        't_bearing_delta', 't_rear_bearing_delta', 't_stator_delta',
        't_bearing_diff', 't_stator_bearing_diff',
        'apparent_power_kva', 'reactive_power_ratio',
        't_bearing_delta_roc', 't_stator_roc',
    ],
    'pitch_bat': [
        'motor_current_axis_1_a', 'motor_current_axis_2_a', 'motor_current_axis_3_a',
        'blade_angle_pitch_position_a', 'blade_angle_pitch_position_b', 'blade_angle_pitch_position_c',
        # NUEVO: deltas motor-ambiente en lugar de temperatura absoluta
        't_motor1_vs_ambient', 't_motor2_vs_ambient', 't_motor3_vs_ambient',
        'power_kw', 'wind_speed_ms',
        'pitch_asymmetry', 'blade_angle_mean', 'motor_current_imbalance',
    ],
}

# Generar y guardar
for family, sensors in FAMILY_SENSORS.items():
    t0 = time.time()
    print(f'\nGenerando {family}...')
    feats  = make_rolling_features_v2(df, sensors, WINDOWS, baseline_mean, baseline_p90)
    tcols  = [f'hours_to_{family}', f'is_pre_{family}']
    output = pd.concat([df[['timestamp']], df[tcols], feats], axis=1)
    output.to_parquet(os.path.join(silver_dir, f'features_{family}_v3.parquet'), index=False)
    print(f'  ✅ features_{family}_v3.parquet | shape={output.shape} | {time.time()-t0:.0f}s')

print('\n✅ COMPLETADO — archivos: features_*_v3.parquet')



Generando yaw_cable...
  ✅ features_yaw_cable_v3.parquet | shape=(210384, 207) | 33s

Generando brake_hydro...
  ✅ features_brake_hydro_v3.parquet | shape=(210384, 241) | 38s

Generando generator...
  ✅ features_generator_v3.parquet | shape=(210384, 360) | 58s

Generando pitch_bat...
  ✅ features_pitch_bat_v3.parquet | shape=(210384, 241) | 38s

✅ COMPLETADO — archivos: features_*_v3.parquet
